In [ ]:
import pandas as pd

df = pd.read_excel('dados/reserva.xlsx')

#  identifica valores ausente, atraves de variavel booleana
df.isna()
#conta de valores ausentes
df.isna().sum()

#identificar valores presentes
df.notna()
#conte de valores presentes
df.notna().sum()

#seleciona somente reservas com informações
df[df['Canal'].notna()]


# fillna preenche valores ausentes
df['Canal'] = df['Canal'].fillna('Não informado')

# remover observações incompletas
# ATENCAO: UMA RESERVA VALIDA PODE SER EXCLUIDA APENAS PORQUE UM CAMPO NAO ESTAVA PREENCHIDO

# df.dropna()

# Podemos restringir a remoção a colunas específicas:
df.dropna(subset=['Valor'])



In [ ]:


import pandas as pd 

df = pd.read_excel('dados/reserva.xlsx', sheet_name='Reservas')

display(df.head())
display(df.info())


# 1. Identificar a quantidade de valores ausentes em cada coluna.

valores_ausentes = df.isna().sum()
display(valores_ausentes)

# 2. Calcular a porcentagem de dados ausentes por coluna. Utilize a quantidade de linhas como denominador.
pct_ausentes = (valores_ausentes * 100 ) / len(df)
display(pct_ausentes)

# 3. Criar um DataFrame contendo somente as reservas cujo canal de aquisição não foi informado.

df_sem_canal = df[df['Canal'].isna()]
display(df_sem_canal)

# 4.

copia_df_ausente = df.copy()
copia_df_ausente['Canal']  = copia_df_ausente['Canal'].fillna('Não informado')
display(copia_df_ausente)

# 5. Criar outra cópia contendo apenas reservas com Valor_R$ preenchido, sem modificar o original.

copia_df_valor = df[df['Valor_R$'].notna()].copy()
display(copia_df_valor)

display('Antes', len(df))
display('Depois', len(copia_df_valor))
display('Removidos', len(df) - len(copia_df_valor))




In [ ]:
# aula 2 duplicatas e tipos da dados

# identificando duplicatas

df.duplicated()

# Também podemos identificar repetições com base em uma coluna específica:
df.duplicated(subset=['Reserva_ID'])

#Depois de identificar o problema, podemos remover os registros repetidos:

df_limpo = df.drop_duplicates(
    subset=['Reserva_ID'],
    keep='first' # preserva a primeira ocorrencia, keep='last(preserva a ultima), keep=false(deleta todas)
)

# tipos de dados

#astype() serve par converter o tipo de uma coluna ou array, Exemplo:

#df['ID_Reserva'] = df['ID_Reserva'].astype('string')


#, converter valores monetários armazenados como texto diretamente para float pode gerar erros caso existam caracteres inválidos.

#Nesse caso, podemos utilizar pd.to_numeric():

df['Valor_R$'] = pd.to_numeric(
    df['Valor_R$'],
    errors='coerce' # transforma valores não convertiveis em nan
)




In [ ]:
import pandas as pd 


df = pd.read_excel('dados/reserva.xlsx', sheet_name='Reservas')

display(df.head())

duplicadas = df.duplicated().sum()
display(duplicadas)

reserva_duplicada = df.duplicated(subset=['Reserva_ID'])

display(reserva_duplicada)

copia = df.copy()

copia = copia.drop_duplicates(
    subset=['Reserva_ID'],
    keep='first'
    )
display(copia)

display('Antes', len(df['Reserva_ID']))
display('Depois', len(copia['Reserva_ID']))


numeros_disponiveis = copia['Valor_R$'].notna()
valor_preenchidos = copia['Valor_R$'].dropna()

display(numeros_disponiveis)
display(valor_preenchidos)

receita_antes = df['Valor_R$'].sum()
display(receita_antes)
receita_total = valor_preenchidos.sum()
display('Receita total em valores preenchidos: ', receita_total)
display('diferença', receita_antes - receita_total)

1   isna(), notna(), fillna() e dropna()	Dados ausentes
2	duplicated(), drop_duplicates(), astype() e to_numeric()	Duplicatas e tipos
3	merge() e concat()	Integração de tabelas
4	pivot_table() e melt()	Reorganização dos dados
5	Projeto integrador	Limpeza, integração e relatório comercial

In [ ]:
# Integração de dados com merge

resultado = pd.merge(
    reservas,
    clientes,
    on='Cliente_ID',
    how='left',
    validate='many_to_one'
)




In [ ]:
# Exercício Merge

import pandas as pd 

df_reservas = pd.read_excel('dados/reserva.xlsx', sheet_name='Reservas')
df_clientes = pd.read_excel('dados/reserva.xlsx', sheet_name='Clientes')

df_reservas = df_reservas.drop_duplicates(
    subset=['Reserva_ID'],
    keep='first'
    ).copy()


#display(df_reservas.duplicated(subset=['Reserva_ID']))
display(df_reservas.columns.to_list())
display(df_clientes.columns.to_list())

relacao = pd.merge(
    df_reservas,
    df_clientes,
    on='Cliente_ID',
    how='left',
    validate='many_to_one',
    indicator=True
)

inner = pd.merge(
    df_reservas,
    df_clientes,
    on='Cliente_ID',
    how='inner',
    validate='many_to_one',
    
)

display(relacao)
display(len(df_reservas))
display(len(relacao))
display(len(inner))

cli_nan = relacao[relacao['Cliente_ID'].isna()]
reserva_sem_cliente = relacao[
    relacao['_merge'] == 'left_only'
]
display(reserva_sem_cliente.value_counts())
display(cli_nan)


In [ ]:
# integrando planilha pagamentos
# a proxima etapa é a junção da tabela pagamentos.
# cada reserva pode ter mais de um pagamentos, que na juncao podem ficar duplicados, para que isso não aconteça realizamos um group_by antes da junção.

df_pagamentos = pd.read_excel('dados/reserva.xlsx', sheet_name='Pagamentos')

display(df_pagamentos.columns.to_list())
display(df_pagamentos.info())
display(df_pagamentos.head())


# quantos pagamento cada reserva recebeu:
contagem_pagamentos = df_pagamentos.groupby(
    'Reserva_ID'
).size()
#O método .size() conta a quantidade de linhas em cada grupo, incluindo linhas com valores ausentes em outras colunas.
display(contagem_pagamentos)

pagamentos_por_reserva = df_pagamentos.groupby(
    'Reserva_ID', as_index=False #mantem a coluna selecionada como comum ao inves de df
)['Valor_Pago_R$'].sum() # essa funcao realiza um agrupamento e soma do valor pago por id de reserva


display(pagamentos_por_reserva.head())

reservas_pagamentos = pd.merge(
    df_reservas,
    pagamentos_por_reserva,
    on='Reserva_ID',
    how='left',
    validate='one_to_one',
    indicator=True #cira a coluna merge que indentifica a origem de cada correspondencia
)

display(reservas_pagamentos)

reservas_sem_pagamentos = reservas_pagamentos[
    reservas_pagamentos['_merge'] == 'left_only'
]

display(reservas_sem_pagamentos)

import numpy as np

valor = reservas_pagamentos['Valor_R$']
pago = reservas_pagamentos['Valor_Pago_R$']

reservas_pagamentos['Comparação valor'] = np.select(
    [ 
    valor == pago,
    valor > pago,
    valor < pago
    ],
    [
        'Pago',
        'Montante a pagar',
        'Débito em conta'
    ],
    default='Sem informações'
)



        


display(reservas_pagamentos)


In [29]:
# avançando de integração para reorganização de dados

# pivot_table() permite apresentar os resultados diretamente em uma tabela cruzada.  melt() tabela dinâmica novamente em um formato com três colunas.

# aggfunc define a operacao estatística

# exercício relatório comercial da pousada

import pandas as pd

df_reservas = pd.read_excel('dados/reserva.xlsx', sheet_name='Reservas')
df_pagamentos = pd.read_excel('dados/reserva.xlsx', sheet_name='Pagamentos')


display(df_pagamentos)
display(df_reservas.columns.to_list())
display(df_reservas.head())

# Etapa 1 — Preparação: remover reservas duplicadas e identificar canais ou categorias ausentes.

display(df_reservas.duplicated())
df_reservas = df_reservas.drop_duplicates(
    subset='Reserva_ID',
    keep='first'
)

# canais ou categorias ausentes
display(df_reservas[df_reservas['Canal'].isna()])

# Etapa 2 — Tabela dinâmica: calcular o faturamento por canal e categoria de acomodação utilizando pivot_table().

df_metodo = df_pagamentos.merge(
    df_reservas[['Reserva_ID', 'Canal']],
    on='Reserva_ID',
    how='left',
    validate='many_to_one',
    indicator=True    
)

faturamento_canal = df_metodo.pivot_table(
    index='Canal',
    columns='Metodo',
    values='Valor_Pago_R$',
    aggfunc='sum',
    fill_value=0
    
)


display(faturamento_canal)

display(df_metodo['_merge'].value_counts())


,Pagamento_ID,Reserva_ID,Metodo,Valor_Pago_R$,Status_Pagamento
0,5001,1001,Pix,290.0,Pendente
1,5002,1002,Cartao,1130.0,Pago
2,5003,1003,Transferencia,1530.0,Pago
3,5004,1004,Pix,1820.0,Pago
4,5005,1005,Cartao,2000.0,Pendente
5,5007,1007,Pix,NaN,Pago
6,5008,1008,Cartao,NaN,Pago
7,5009,1009,Transferencia,2040.0,Pendente
8,5010,1010,Pix,2275.0,Pago
9,5011,1011,Cartao,400.0,Pago


['Reserva_ID',
 'Cliente_ID',
 'Data_Entrada',
 'Canal',
 'Diarias',
 'Valor_R$',
 'Status']

,Reserva_ID,Cliente_ID,Data_Entrada,Canal,Diarias,Valor_R$,Status
0,1001,201,2026-01-05,WhatsApp,1.0,290.0,Confirmada
1,1002,202,2026-01-13,Instagram,2.0,1130.0,Concluida
2,1003,203,2026-01-21,Booking,3.0,1530.0,Cancelada
3,1004,204,2026-01-29,Telefone,4.0,1820.0,Confirmada
4,1005,205,2026-02-06,NaN,5.0,2000.0,Concluida


0     False
1     False
2     False
3     False
4     False
5     False
6     False
7     False
8     False
9     False
10    False
11    False
12    False
13    False
14    False
15    False
16    False
17    False
18    False
19    False
20    False
21    False
22    False
23    False
24    False
25    False
26    False
27    False
28    False
29    False
30    False
31    False
32    False
33    False
34    False
35    False
36    False
37    False
38    False
39    False
40    False
41    False
42     True
dtype: bool

,Reserva_ID,Cliente_ID,Data_Entrada,Canal,Diarias,Valor_R$,Status
4,1005,205,2026-02-06,NaN,5.0,2000.0,Concluida
15,1016,216,2026-05-05,NaN,1.0,455.0,Confirmada
26,1027,209,2026-08-01,NaN,2.0,1020.0,Cancelada
37,1038,202,2026-01-06,NaN,3.0,1695.0,Concluida


Metodo,Cartao,Pix,Transferencia
Canal,,,
Booking,1200.0,2235.0,4095.0
Instagram,3060.0,580.0,1380.0
Site,4825.0,4550.0,4275.0
Telefone,2260.0,4800.0,5460.0
WhatsApp,1365.0,580.0,855.0


_merge
both          40
left_only      0
right_only     0
Name: count, dtype: int64

In [30]:
# Etapa 3 — Indicadores: identificar o canal de maior faturamento e a acomodação com maior receita.

receitas_por_canal = df_metodo.groupby(
    ['Canal'], as_index=False
)['Valor_Pago_R$'].sum()

display(receitas_por_canal)
maior_receitas_por_canal = receitas_por_canal.idxmax()
display(receitas_por_canal.loc[maior_receitas_por_canal])

receitas_por_metodo = df_metodo.groupby(
    ['Metodo'], as_index=False
)['Valor_Pago_R$'].sum()

display('Receitas por Método:', receitas_por_metodo)
display(receitas_por_metodo.max())

,Canal,Valor_Pago_R$
0,Booking,7530.0
1,Instagram,5020.0
2,Site,13650.0
3,Telefone,12520.0
4,WhatsApp,2800.0


,Canal,Valor_Pago_R$
4,WhatsApp,2800.0
2,Site,13650.0


'Receitas por Método:'

,Metodo,Valor_Pago_R$
0,Cartao,16405.0
1,Pix,13200.0
2,Transferencia,17085.0


Metodo           Transferencia
Valor_Pago_R$          17085.0
dtype: object

In [31]:
# Etapa 4 — Transformação: utilizar reset_index() e melt() para converter a tabela dinâmica em formato longo.

df_metodo_longo = faturamento_canal.reset_index().melt(
    id_vars='Canal',
    var_name='Metodo',
    value_name='Valor_Pago_R$'
)

display(df_metodo_longo)

,Canal,Metodo,Valor_Pago_R$
0,Booking,Cartao,1200.0
1,Instagram,Cartao,3060.0
2,Site,Cartao,4825.0
3,Telefone,Cartao,2260.0
4,WhatsApp,Cartao,1365.0
5,Booking,Pix,2235.0
6,Instagram,Pix,580.0
7,Site,Pix,4550.0
8,Telefone,Pix,4800.0
9,WhatsApp,Pix,580.0


In [42]:
# Etapa 5 — Validação: comparar a soma da tabela dinâmica com o faturamento dos registros únicos, verificando se os totais correspondem.

total_pivot = faturamento_canal.to_numpy().sum()

total_pagamentos = df_pagamentos['Valor_Pago_R$'].sum()

print('Tabela dinâmica:', total_pivot)
print('Pagamentos originais:', total_pagamentos)
print('Diferença:', total_pagamentos - total_pivot)

Diferença = total_pagamentos - total_pivot

display(df_pagamentos)
display(df_reservas)

canais_vazios = df_metodo['Canal'].isna()
display(df_metodo[canais_vazios])
valor_canal_vazio = df_metodo.loc[canais_vazios, 'Valor_Pago_R$'].sum()

display(valor_canal_vazio)

variacao = valor_canal_vazio == Diferença

print('O valor de variação vem de canais NaN: ', variacao)

Tabela dinâmica: 41520.0
Pagamentos originais: 46690.0
Diferença: 5170.0


,Pagamento_ID,Reserva_ID,Metodo,Valor_Pago_R$,Status_Pagamento
0,5001,1001,Pix,290.0,Pendente
1,5002,1002,Cartao,1130.0,Pago
2,5003,1003,Transferencia,1530.0,Pago
3,5004,1004,Pix,1820.0,Pago
4,5005,1005,Cartao,2000.0,Pendente
5,5007,1007,Pix,NaN,Pago
6,5008,1008,Cartao,NaN,Pago
7,5009,1009,Transferencia,2040.0,Pendente
8,5010,1010,Pix,2275.0,Pago
9,5011,1011,Cartao,400.0,Pago


,Reserva_ID,Cliente_ID,Data_Entrada,Canal,Diarias,Valor_R$,Status
0,1001,201,2026-01-05,WhatsApp,1.0,290.0,Confirmada
1,1002,202,2026-01-13,Instagram,2.0,1130.0,Concluida
2,1003,203,2026-01-21,Booking,3.0,1530.0,Cancelada
3,1004,204,2026-01-29,Telefone,4.0,1820.0,Confirmada
4,1005,205,2026-02-06,NaN,5.0,2000.0,Concluida
5,1006,206,2026-02-14,WhatsApp,1.0,345.0,Cancelada
6,1007,207,2026-02-22,Instagram,2.0,580.0,Confirmada
7,1008,208,2026-03-02,Booking,3.0,NaN,Concluida
8,1009,209,2026-03-10,Telefone,4.0,2040.0,Cancelada
9,1010,210,2026-03-18,Site,NaN,2275.0,Confirmada


,Pagamento_ID,Reserva_ID,Metodo,Valor_Pago_R$,Status_Pagamento,Canal,_merge
4,5005,1005,Cartao,2000.0,Pendente,NaN,both
14,5016,1016,Pix,455.0,Pago,NaN,both
24,5027,1027,Transferencia,1020.0,Pago,NaN,both
35,5038,1038,Cartao,1695.0,Pago,NaN,both


np.float64(5170.0)

O valor de variação vem de canais NaN:  True
